# AgentCore Gateway 대상에 Authorization code grant 유형의 아웃바운드 OAuth 흐름 사용

## 개요
아웃바운드 인증/권한 부여를 사용하면 Amazon Bedrock AgentCore Gateway가 인바운드 권한 부여 과정에서 인증 및 권한 부여된 사용자를 대신하여 Gateway 대상에 안전하게 액세스할 수 있습니다. AgentCore Gateway는 권한 부여 없음, IAM 기반 아웃바운드 권한 부여, OAuth, API key 등 여러 유형의 아웃바운드 권한 부여를 지원합니다.

다음과 같은 OAuth 권한 부여 유형을 사용할 수 있습니다.
* **Client credentials grant** – 시스템 간 인증(2-legged OAuth라고도 함)입니다. 클라이언트 애플리케이션은 사용자가 아닌 애플리케이션을 대신하여 리소스에 액세스합니다.
* **Authorization code grant (신규)** – 사용자 위임 액세스(3-legged OAuth라고도 함)입니다. 사용자는 클라이언트 애플리케이션이 자신을 대신하여 리소스에 액세스하도록 동의합니다.

Model Context Protocol (MCP) 사양의 **version 2025-11-25**부터 MCP는 [URL Mode Elicitation](https://blog.modelcontextprotocol.io/posts/2025-11-25-first-mcp-anniversary/#url-mode-elicitation-secure-out-of-band-interactions)을 지원합니다. URL mode elicitation을 사용하면 사용자를 브라우저의 적절한 OAuth 흐름(또는 다른 자격 증명 획득 흐름)으로 안내할 수 있습니다. 이때 클라이언트는 사용자가 입력한 자격 증명을 전혀 확인하지 않으면서 사용자가 안전하게 인증할 수 있습니다. 이후 자격 증명은 서버에서 직접 관리되며, 클라이언트는 서버에 대한 자체 권한 부여 흐름만 처리하면 됩니다.

이제 AgentCore Gateway는 2025-11-25를 지원 MCP 버전으로 지정하여 생성한 Gateway에서 **Authorization code grant**를 Outbound Auth로 지원합니다. Authorization code grant는 Google(이메일, 캘린더 등), Github(리포지토리, pull request 등), Linkedin(사용자 프로필, 게시물 등) 또는 내부 도구에 사용자를 대신하여 액세스할 때 유용합니다. Authorization code grant 흐름은 사용자 자격 증명을 서드 파티 애플리케이션에 노출하지 않아 사용자 개인정보 보호를 우선합니다.

이 튜토리얼에서는 Authorization code grant를 사용하고 Linkedin 도구를 대상으로 하는 AgentCore Gateway를 생성합니다. 생성한 AgentCore Gateway를 사용하면 사용자 정보 가져오기, 게시물 읽기/요약, 새 게시물 작성 등 사용자를 대신하여 Linkedin 작업을 자동화하는 에이전트를 구축할 수 있습니다.


## 적절한 인증/권한 부여 패턴 선택
에이전트 인증/권한 부여 전략을 설계할 때는 다음 요소를 고려하여 가장 적합한 패턴을 결정합니다.

| **요소** | **OAuth 2.0 authorization code grant(사용자 위임 액세스)** | **OAuth 2.0 client credentials grant(시스템 간 인증)** |
|------------|----------------------------------------------------------------|-----------------------------------------------------------------------------|
| **데이터 소유권** | 사용자별 데이터(이메일, 문서, 개인 캘린더) | 시스템 또는 조직 소유 데이터(분석, 로그, 공유 리소스) |
| **사용자 상호 작용** | 사용자가 참여하여 동의를 제공할 수 있음 | 사용자 상호 작용이 필요하지 않거나 불가능함 |
| **작업 시점** | 대화형 실시간 작업 | 백그라운드, 예약 또는 배치 작업 |
| **권한 범위** | 사용자와 사용자의 동의 선택에 따라 권한이 달라짐 | 에이전트 수준에서 정의된 일관된 권한 |

## Authorization code grant의 주요 특징

* 권한 부여 프롬프트를 통한 명시적인 사용자 동의 필요
* 사용자별 데이터 및 리소스에 대한 액세스 제공
* 에이전트 자격 증명과 사용자 권한 부여를 명확하게 분리
* 에이전트가 액세스할 수 있는 데이터를 제한하는 세분화된 범위 지원

### 튜토리얼 아키텍처

<div style="text-align:center">
    <img src="images/outbound_auth_3lo.png" width="90%"/>
</div>


### 튜토리얼 세부 정보

| 정보                | 세부 정보                                                                |
|:--------------------|:-------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                    |
| 에이전트 유형       | MCP Client                                                               |
| 에이전트 프레임워크 | MCP SDK                                                                  |
| LLM 모델            | N/A                                                                      |
| 튜토리얼 구성 요소  | Linkedin 대상을 사용하는 AgentCore Gateway(OAuth 2.0 authorization code grant 사용)            |
| 튜토리얼 분야       | 여러 분야에 적용 가능                                                    |
| 예제 복잡도         | 중간                                                                     |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 Boto3                              |
| 자격 증명 공급자 | 유형: OAuth2 - Linkedin Provider                                         |


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS 자격 증명
* UV

In [ ]:
# 현재 디렉터리의 requirements 파일에서 설치
!uv pip install --system --force-reinstall --no-cache-dir -r requirements.txt --quiet

In [ ]:
# 필수 라이브러리를 가져오고 리소스 이름에 사용할 고유한 타임스탬프를 생성합니다.

import boto3
import json
import time
import os
import sys
import requests
from botocore.exceptions import ClientError
from datetime import datetime

print("✓ Libraries imported")

# 고유한 이름을 위한 타임스탬프 생성
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
print(f"Using timestamp: {timestamp}")

REGION = os.environ["AWS_REGION"]

gateway_target_name = f"mcp-target-{timestamp}"

In [ ]:
# 유틸리티 가져오기 및 로깅 구성

# 현재 스크립트의 디렉터리 확인
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우(예: Jupyter) 대체 경로 사용

# utils.py가 있는 디렉터리로 이동(한 수준 위)
utils_dir = os.path.abspath(os.path.join(current_dir, ".."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

# utils 가져오기
import utils

# 로깅 설정
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

logging.getLogger("strands").setLevel(logging.INFO)

print("✓ Logging configured, utils imported")

## Amazon Cognito를 IDP로 사용하여 Inbound Auth 구성

App client가 포함된 Cognito Userpool을 프로비저닝합니다. Amazon Cognito를 사용하여 배포된 MCP 서버에 액세스하기 위한 JWT token을 제공합니다.

In [ ]:
USER_POOL_NAME = f"agentcore-gateway-authcode-pool-{timestamp}"
RESOURCE_SERVER_ID = f"agentcore-gateway-authcode-id-{timestamp}"
RESOURCE_SERVER_NAME = f"agentcore-gateway-authcode-name-{timestamp}"
CLIENT_NAME = f"agentcore-gateway-authcode-client-{timestamp}"

# 범위는 이번 실행의 현재 gateway_target_name을 기준으로 함
SCOPES = [
    # MCP 대상에 대한 전체 액세스
    {
        "ScopeName": gateway_target_name,
        "ScopeDescription": "Full access to all tools in MCP target",
    }
]

# Cognito 형식의 전체 범위 문자열: "<resource-server-id>/<scope-name>"
scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(
    cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
)
print(f"Client ID: {gw_client_id}")

# 이후 Gateway authorizer와 utils.get_token에서 사용할 Discovery URL
gw_cognito_discovery_url = (
    f"https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration"
)
print(gw_cognito_discovery_url)

## Linkedin을 Outbound OAuth2의 credential provider로 구성

### 1단계: LinkedIn Client Id 및 Client Secret 발급

LinkedIn API를 사용하여 Auth code grant 흐름을 테스트합니다. LinkedIn API를 호출할 access token을 발급받으려면 먼저 LinkedIn ClientId와 ClientSecret이 필요합니다. 발급 방법은 다음과 같습니다.

* https://developer.linkedin.com/으로 이동합니다.
* create app을 클릭하고 앱 이름을 입력합니다.
* 필수 단계로 LinkedIn 페이지를 생성해야 합니다. For Business를 클릭하면 페이지를 생성하는 옵션이 표시됩니다. 테스트용 페이지를 생성합니다. 
* App을 생성하면 개발자 포털의 My Apps 섹션에서 확인할 수 있습니다.
* App의 Auth 섹션에서 access token 발급에 사용할 clientId와 clientSecret을 확인할 수 있습니다. 
* App의 Products 섹션에서 액세스할 API를 활성화해야 합니다. “Sign In with LinkedIn using OpenID Connect”의 “Request Access”를 클릭합니다.
* 이렇게 하면 사용자 프로필 정보를 가져오는 API가 활성화됩니다.

<div style="text-align:center">
    <img src="images/linkedin-oauth1.png" width="90%"/>
</div>

<div style="text-align:center">
    <img src="images/linkedin-oauth2.png" width="90%"/>
</div>

참고: Oauth 2.0 settings 섹션에는 Authorized Redirect URLs를 입력해야 합니다. 다음 단계에서 credential provider를 생성한 후 이 섹션으로 돌아옵니다.

### 2단계: AgentCore Identity로 Linkedin Credential provider 생성
Amazon Bedrock AgentCore Identity는 인바운드 및 아웃바운드 인증 모두에 대해 관리형 OAuth 2.0 지원 provider를 제공합니다. 각 provider는 특정 서비스 또는 자격 증명 시스템에 필요한 인증 프로토콜, endpoint 구성 및 자격 증명 형식을 캡슐화합니다. 이 서비스는 개발 작업을 줄이기 위해 authorization server endpoint와 provider별 parameter가 사전 구성된 Google, GitHub, Slack, Salesforce 등의 인기 서비스용 내장 provider를 제공합니다. 이러한 provider는 서로 다른 OAuth 2.0 구현, API 인증 체계, token 형식의 복잡성을 추상화하여 에이전트에 통합 인터페이스를 제공하는 동시에 기본 프로토콜의 차이와 예외 상황을 처리합니다.

이 튜토리얼에서는 Linkedin을 내장 provider로 사용하여 credential provider를 생성합니다.

**앞에서 기록한 Linkedin Client Id와 Client Secret을 입력하세요.**

In [ ]:
target_client_id = "<clientid>"  # https://developer.linkedin.com/에서 발급받은 client id로 교체
target_client_secret = "<clientsecret>"  # https://developer.linkedin.com/에서 발급받은 client secret으로 교체

target_cred_provider_name = f"ac-gateway-mcp-server-identity-authcode-{timestamp}"

identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

print(f"Deleting the credential provider with name {target_cred_provider_name} if it exists already")
try:
    delete_resp = identity_client.delete_oauth2_credential_provider(name=target_cred_provider_name)
    print("Existing credential provider found and deleted. Proceeding with re-creation")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFound":
        print("Existing credential provider with same name does not exist. Proceeding with creation")
    else:
        raise Exception(f"Credential provider deletion failed: {e.response['Error']['Message']}")

linkedin_cred_provider = identity_client.create_oauth2_credential_provider(
    name=target_cred_provider_name,
    credentialProviderVendor="LinkedinOauth2",
    oauth2ProviderConfigInput={
        "linkedinOauth2ProviderConfig": {
            "clientId": target_client_id,
            "clientSecret": target_client_secret,
        }
    },
)

target_cred_provider_arn = linkedin_cred_provider["credentialProviderArn"]
target_callback_url = linkedin_cred_provider["callbackUrl"]
print("Outbound OAuth2 Credential Provider ARN:", target_cred_provider_arn)
print("Please register the following callback URL with linkedin ", target_callback_url)

### 3단계: linkedin에 callback URL 등록
앞 셀의 출력으로 받은 callback URL을 Linkedin App에 등록합니다.

<div style="text-align:center">
    <img src="images/linkedin-oauth3.png" width="90%"/>
</div>

## Cognito를 inbound auth로 사용하는 AgentCore Gateway 생성
Cognito를 inbound Auth provider로 사용하는 AgentCore Gateway를 생성합니다.

In [ ]:
def create_gateway_with_authcode():
    iam_client = boto3.client("iam", region_name=REGION)
    gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

    role_name = f"BedrockAgentCoreGatewayRole-{timestamp}"
    gateway_name = f"gateway-authcode-{timestamp}"

    # Gateway용 IAM role 생성
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }

    try:
        iam_response = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="IAM role for Bedrock Agent Core Gateway with 3LO",
        )

        role_arn = iam_response["Role"]["Arn"]
        print(f"Gateway IAM role created: {role_arn}")

        # role에 admin policy 연결
        iam_client.attach_role_policy(RoleName=role_name, PolicyArn="arn:aws:iam::aws:policy/AdministratorAccess")
        print("Admin policy attached to Gateway IAM role")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            iam_response = iam_client.get_role(RoleName=role_name)
            role_arn = iam_response["Role"]["Arn"]
            print(f"IAM role already exists with Arn: {role_arn}. Using the same")
        else:
            raise Exception(f"IAM Role creation failed: {e.response['Error']['Message']}")

    print("Creating gateway with Auth Code grant...")
    gateway_response = gateway_client.create_gateway(
        name=gateway_name,
        protocolType="MCP",
        protocolConfiguration={
            "mcp": {
                "supportedVersions": ["2025-03-26", "2025-11-25"],
                "searchType": "SEMANTIC",
            }
        },
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw_cognito_discovery_url,
                "allowedClients": [gw_client_id],
            }
        },
        roleArn=role_arn,
    )

    print("Gateway create response:", gateway_response)

    gateway_id = gateway_response["gatewayId"]
    gateway_url = gateway_response["gatewayUrl"]
    print(f"Gateway with Auth code grant created: {gateway_id}")

    # Gateway가 준비될 때까지 대기
    print("Waiting for gateway to be ready...")
    while True:
        status_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)

        current_status = status_response["status"]
        print(f"Gateway status: {current_status}")
        if current_status == "READY":
            print(f"Final gateway details: {status_response}")
            break

        time.sleep(10)

    print("Gateway is now ready")
    return gateway_id, gateway_url, role_name


gateway_id, gateway_url, gateway_role_name = create_gateway_with_authcode()
print(f"\n✅ Gateway creation completed: Gateway Id {gateway_id}")
print(f"Gateway Url: {gateway_url}")

## AgentCore Gateway로 대상 생성
**/userInfo**용 도구가 포함된 Linkedin OpenAPI spec을 사용하여 AgentCore Gateway에 대상을 생성합니다.

In [ ]:
linkedin_openapi_spec = {
    "openapi": "3.0.0",
    "info": {"title": "LinkedIn UserInfo API", "version": "2.0.0"},
    "servers": [{"url": "https://api.linkedin.com/v2"}],
    "paths": {
        "/userinfo": {
            "get": {
                "operationId": "getUserInfo",
                "summary": "Get User Information",
                "security": [{"BearerAuth": []}],
                "responses": {
                    "200": {
                        "description": "Successful response",
                        "content": {
                            "application/json": {
                                "schema": {"$ref": "#/components/schemas/UserInfo"},
                                "example": {
                                    "sub": "782bbtaQ",
                                    "name": "John Doe",
                                    "given_name": "John",
                                    "family_name": "Doe",
                                    "picture": "https://media.licdn-ei.com/dms/image/C5F03AQHqK8v7tB1HCQ/profile-displayphoto-shrink_100_100/0/",
                                    "locale": "en-US",
                                    "email": "doe@email.com",
                                    "email_verified": True,
                                },
                            }
                        },
                    }
                },
            }
        }
    },
    "components": {
        "schemas": {
            "UserInfo": {
                "type": "object",
                "properties": {
                    "sub": {"type": "string"},
                    "name": {"type": "string"},
                    "given_name": {"type": "string"},
                    "family_name": {"type": "string"},
                    "picture": {"type": "string"},
                    "locale": {"type": "string"},
                    "email": {"type": "string"},
                    "email_verified": {"type": "boolean"},
                },
            }
        },
        "securitySchemes": {"BearerAuth": {"type": "http", "scheme": "bearer"}},
    },
}

In [ ]:
DEFAULT_RETURN_URL = (
    "http://localhost:3021"  # 테스트용 URL을 기본값으로 사용하며 JSON RPC 호출 시 재정의
)


def create_linkedin_target(gatewayId, provider_arn):
    gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
    credentialProviderConfig = {
        "credentialProviderType": "OAUTH",
        "credentialProvider": {
            "oauthCredentialProvider": {
                "providerArn": provider_arn,
                "grantType": "AUTHORIZATION_CODE",
                "defaultReturnUrl": DEFAULT_RETURN_URL,
                "scopes": ["openid", "profile", "email"],
            }
        },
    }
    target_config = {"mcp": {"openApiSchema": {"inlinePayload": json.dumps(linkedin_openapi_spec)}}}
    try:
        response = gateway_client.create_gateway_target(
            name="LinkedInAuthCode",
            description="Target created for testing",
            credentialProviderConfigurations=[credentialProviderConfig],
            targetConfiguration=target_config,
            gatewayIdentifier=gatewayId,
        )
        targetId = response["targetId"]
        print(f"Created Linkedin target {targetId} for gateway {gatewayId}")
        return targetId
    except Exception as e:
        print(e)


gateway_target_id = create_linkedin_target(gatewayId=gateway_id, provider_arn=target_cred_provider_arn)

## OAuth2 Authorization URL 세션 바인딩 프로세스

Gateway를 생성했으므로 다음 단계로 진행하기 전에 세션 바인딩 프로세스를 살펴보겠습니다.

OAuth2 authorization URL 세션 바인딩 프로세스는 OAuth2 권한 부여 세션이 AgentCore Identity에서 인증된 사용자와 올바르게 연결되도록 보장하는 핵심 보안 메커니즘입니다. 이 프로세스는 세션 하이재킹을 방지하고 의도한 사용자에게만 OAuth token이 부여되도록 합니다.

참고: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html

### 세션 바인딩 작동 방식
<div style="text-align:center">
    <img src="images/identity-session-binding.png" width="90%"/>
</div>

1. 에이전트 호출 – 에이전트를 시작한 사용자가 자신이 소유한 애플리케이션 또는 리소스에 액세스하려는 경우, 에이전트 코드 또는 MCP client가 GetResourceOauth2Token API를 호출하여 authorization URL을 가져옵니다.

2. authorization URL 생성 – AgentCore Identity는 사용자가 이동하여 액세스에 동의할 수 있는 authorization URL과 session URI를 생성합니다.

3. 권한 부여 및 access token 발급 – 사용자는 authorization URL로 이동하여 에이전트가 자신의 리소스에 액세스하는 데 동의합니다. 그러면 AgentCore Identity는 권한 부여 요청을 시작한 사용자 정보와 함께 사용자의 브라우저를 애플리케이션의 HTTPS endpoint로 리디렉션합니다. 이때 HTTPS 애플리케이션 endpoint는 에이전트를 시작한 사용자와 현재 애플리케이션에 로그인한 사용자가 같은지 확인합니다. 두 사용자가 일치하면 애플리케이션 endpoint가 CompleteResourceTokenAuth를 호출하고, AgentCore Identity가 access token을 가져와 저장합니다.

4. access token 발급을 위한 에이전트 재호출 – 애플리케이션이 유효한 응답을 반환하면 에이전트 애플리케이션은 처음에 해당 사용자를 위해 요청했던 OAuth2.0 access token을 가져올 수 있습니다. 사용자가 일치하지 않으면 애플리케이션은 아무 작업도 하지 않거나 해당 시도를 기록합니다.

AgentCore Identity는 애플리케이션 endpoint가 사용자 자격 증명을 확인하도록 하여, 에이전트 애플리케이션이 권한 부여 요청을 시작한 사용자와 액세스에 동의한 사용자가 항상 같은지 확인할 수 있게 합니다.

### 실행 환경을 인식하는 OAuth2 Callback Server

이 튜토리얼에서는 실행 환경에 따라 자동으로 조정되는 `oauth2_callback_server.py`를 사용합니다.

#### **로컬 개발:**
- **외부 Callback URL**: `http://localhost:9090/oauth2/callback` (브라우저에서 액세스 가능)
- **내부 통신**: `http://localhost:9090` (Notebook ↔ server)
- **Server Binding**: `127.0.0.1` (localhost에서만 액세스할 수 있어 안전함)

#### **SageMaker Workshop Studio:**
- **외부 Callback URL**: `https://<domain>.studio.<region>.sagemaker.aws/proxy/9090/oauth2/callback` (proxy를 통해 브라우저에서 액세스 가능)
- **내부 통신**: `http://localhost:9090` (동일한 container의 Notebook ↔ server)
- **Server Binding**: `0.0.0.0` (SageMaker proxy에서 server에 연결 가능)

OAuth2 callback server는 `/opt/ml/metadata/resource-metadata.json`의 존재 여부를 확인하여 환경을 자동으로 감지하고 그에 맞게 구성됩니다.

### oauth2_callback_server.py의 기능

1. **로컬 FastAPI Server 실행** (port 9090)
   - 상태 확인을 위한 `/ping` endpoint 제공
   - 사용자 token을 저장하는 `/userIdentifier/token` endpoint 제공
   - OAuth redirect를 처리하는 `/oauth2/callback` endpoint 제공

2. **사용자 Token 저장소 관리**
   - Cognito 인증으로 발급받은 사용자의 JWT token 저장
   - OAuth 세션을 올바른 사용자 자격 증명과 연결

3. **OAuth Callback 처리**
   - `session_id` parameter가 포함된 OAuth redirect 수신
   - 세션을 바인딩하기 위해 `CompleteResourceTokenAuth` 호출
   - 흐름을 완료하기 전에 사용자 자격 증명 검증

4. **세션 보안 제공**
   - OAuth 세션이 인증된 사용자에게 바인딩되도록 보장
   - OAuth token에 대한 무단 액세스 방지

5. **환경 감지**
   - 로컬 환경과 SageMaker Studio 환경을 자동으로 감지
   - URL과 server binding을 적절하게 구성

### 보안 고려 사항

OAuth2 세션 바인딩 프로세스에는 다음과 같은 여러 보안 조치가 포함됩니다.
- **URL 검증**: 사전 등록된 callback URL만 허용
- **세션 검증**: token 발급을 완료하기 전에 사용자 세션을 검증해야 함
- **사용자 자격 증명 바인딩**: OAuth 세션을 인증된 사용자에게 명시적으로 바인딩
- **Token 격리**: 각 사용자의 OAuth token을 격리하여 안전하게 보호
- **환경 인식 URL**: 각 환경에 적합한 URL을 자동으로 사용

이러한 종합적인 접근 방식은 로컬 또는 SageMaker Workshop Studio에서 실행되는 다중 사용자 환경에서 OAuth2 흐름이 안전하게 유지되고 올바른 사용자에게 정확히 연결되도록 보장합니다.

---

In [ ]:
# MCP 호출을 위한 헬퍼 함수
def invoke_mcp(
    gatewayUrl,
    access_token,
    tool_params,
    method="tools/call",
    protocol_version="2025-11-25",
):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}",
        "MCP-Protocol-Version": protocol_version,
    }

    payload = {"jsonrpc": "2.0", "id": 24, "method": method, "params": tool_params}

    try:
        response = requests.post(gatewayUrl, headers=headers, json=payload)
        request_id = response.headers.get("x-amzn-requestid") or response.headers.get("x-amz-request-id")
        print("\nAmazon Request ID:", request_id)
        response.raise_for_status()
        print(f"Invoke MCP Status Code: {response.status_code}")
        print("Response:")
        # if method != "tools/list":
        print(json.dumps(response.json(), indent=2))
        return response.json()

    except requests.exceptions.RequestException as e:
        print("Error:", e)
        if hasattr(e, "response") and e.response is not None:
            try:
                print("Error Response:", json.dumps(e.response.json(), indent=2))
            except:
                print("Error Response Text:", e.response.text)
        raise

## MCP Client로 테스트
이제 MCP Client로 AgentCore Gateway를 테스트합니다. 사용자 동의 흐름을 테스트할 수 있도록 **forceAuthentication = True** metadata로 시작합니다. 이 단계를 처음 실행하는 경우에는 **forceAuthentication = False**여도 동작이 동일합니다.

URL을 열어 사용자 동의를 제공하라는 Elicitation Response가 반환됩니다.

동의를 완료하면 callback server가 AgentCore Identity와의 세션 바인딩을 처리합니다.

<div style="text-align:center">
    <img src="images/elicitation-resp.png" width="90%"/>
</div>

In [ ]:
import subprocess
from oauth2_callback_server import (
    store_token_in_oauth2_callback_server,
    wait_for_oauth2_server_to_be_ready,
    get_oauth2_callback_base_url,
)

full_scope = f"{RESOURCE_SERVER_ID}/{gateway_target_name}"
jwt_token = utils.get_token(
    user_pool_id=gw_user_pool_id,
    client_id=gw_client_id,
    client_secret=gw_client_secret,
    scope_string=full_scope,
    REGION=REGION,
)
bearer_token = jwt_token["access_token"]

# OAuth callback server 시작
oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    REGION,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
if not successfully_started_oauth2_server:
    print(
        "Failed to start OAuth2 callback server to handle session binding "
        "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)"
    )
else:
    store_token_in_oauth2_callback_server(bearer_token)

    # 도구 호출 테스트
    CUSTOM_RETURN_URL = get_oauth2_callback_base_url() + "/oauth2/callback"
    print(f"Using callback URL: {CUSTOM_RETURN_URL}")

    _meta = {
        "aws.bedrock-agentcore.gateway/credentialProviderConfiguration": {
            "oauthCredentialProvider": {
                "returnUrl": CUSTOM_RETURN_URL,
                "forceAuthentication": True,
            }
        }
    }

    print("\nTesting force re-authentication")
    resp = invoke_mcp(
        gatewayUrl=gateway_url,
        access_token=bearer_token,
        tool_params={
            "name": "LinkedInAuthCode___getUserInfo",
            "arguments": {"domainName": "integrals-dev-ed"},
            "_meta": _meta,
        },
        method="tools/call",
    )

In [ ]:
# OAuth 흐름을 완료한 후 도구 다시 호출
_meta = {
    "aws.bedrock-agentcore.gateway/credentialProviderConfiguration": {
        "oauthCredentialProvider": {
            "returnUrl": CUSTOM_RETURN_URL,
            "forceAuthentication": False,
        }
    }
}
print("Invoking tool again after completion of oAuth flow")
resp = invoke_mcp(
    gatewayUrl=gateway_url,
    access_token=bearer_token,
    tool_params={
        "name": "LinkedInAuthCode___getUserInfo",
        "arguments": {"domainName": "integrals-dev-ed"},
        "_meta": _meta,
    },
    method="tools/call",
)

In [ ]:
oauth2_callback_server_process.terminate()

## 리소스 정리

다음 셀을 실행하여 이 튜토리얼에서 생성한 모든 리소스를 제거합니다.

1. AgentCore Gateway 및 대상
2. AgentCore Identity 자격 증명 공급자
3. Amazon Cognito User Pool 및 client
4. IAM 역할

In [ ]:
# AgentCore Gateway 및 모든 대상 삭제
print("Step 1: Cleaning up AgentCore Gateway resources...")
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_cleanup = utils.delete_gateway(gateway_client=gateway_client, gatewayId=gateway_id)

# AgentCore Identity Credential Provider 삭제
print("\nStep 2: Cleaning up AgentCore Identity Credential Provider...")
credential_cleanup = identity_client.delete_oauth2_credential_provider(name=target_cred_provider_name)

# Cognito User Pool 삭제
print("\nStep 3: Cleaning up Cognito User Pool...")
cognito_cleanup = utils.delete_cognito_user_pool(user_pool_id=gw_user_pool_id, region=REGION)

# IAM Role 삭제
print("\nStep 4: Cleaning up IAM Role...")
iam_cleanup = utils.delete_iam_role(role_name=f"BedrockAgentCoreGatewayRole-{timestamp}")

## 요약
새로 출시된 authorization code grant 기반 아웃바운드 OAuth 흐름을 Linkedin 도구를 대상으로 사용하는 AgentCore Gateway에서 성공적으로 테스트했습니다.